# 03 — In Season Feature Engineering

This notebook creates the weekly performance features used by the in season projection layer.

It does not modify the original preseason notebooks or their outputs.

The purpose of this notebook is to convert completed regular season games into team level offensive and defensive performance features that can later be blended with the frozen preseason model.

For a selected target week, the notebook:

- Loads only completed games from weeks before the target week
- Builds team level scoring and margin summaries
- Separates offense and defense
- Adds simple opponent adjusted performance features
- Applies early season shrinkage to reduce overreaction to small samples
- Saves a clean feature table for the weekly team strength notebook

This notebook is intentionally conservative after Week 1. One game should inform the model, not redefine it.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the outputs created by:

`02_Weekly_Data_Update.ipynb`

The only value that normally changes week to week is `TARGET_WEEK`.


In [2]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 12
PRIOR_WEEK = TARGET_WEEK - 1

completed_games_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_completed_games.parquet"
)

prior_strength_path = (
    WEEKLY_DATA_DIR
    / f"week_{PRIOR_WEEK:02d}_sequential_team_strength.parquet"
)

print("Using completed games:", completed_games_path)
print("Using entering ratings:", prior_strength_path)

Using completed games: ..\..\data\processed\weekly\week_12_completed_games.parquet
Using entering ratings: ..\..\data\processed\weekly\week_11_sequential_team_strength.parquet


# Load Completed Games

Only games completed before the target week should exist in this file.

That leakage rule was enforced in `02_Weekly_Data_Update.ipynb`.


In [3]:
completed_games = pd.read_parquet(
    completed_games_path
)

completed_games = completed_games.sort_values(
    ["week", "gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Completed games available before Week {TARGET_WEEK}:",
    len(completed_games)
)

display(
    completed_games[
        [
            "week",
            "gameday",
            "away_team",
            "away_score",
            "home_team",
            "home_score"
        ]
    ]
)


Completed games available before Week 12: 163


,week,gameday,away_team,away_score,home_team,home_score
0,1,2026-09-09,NE,10.0,SEA,13.0
1,1,2026-09-10,SF,27.0,LA,7.0
2,1,2026-09-13,ARI,26.0,LAC,14.0
3,1,2026-09-13,ATL,13.0,PIT,20.0
4,1,2026-09-13,BAL,41.0,IND,23.0
...,...,...,...,...,...,...
158,11,2026-11-22,NYJ,27.0,LAC,24.0
159,11,2026-11-22,PIT,16.0,PHI,19.0
160,11,2026-11-22,TB,21.0,DET,31.0
161,11,2026-11-22,TEN,30.0,DAL,51.0


In [4]:
update_games = completed_games[
    completed_games["week"] == PRIOR_WEEK
].copy()

update_games = update_games.sort_values(
    ["gameday", "game_id"]
).reset_index(drop=True)

print(
    f"Week {PRIOR_WEEK} games used to update "
    f"Week {TARGET_WEEK} ratings:",
    len(update_games)
)

display(
    update_games[
        [
            "week",
            "gameday",
            "away_team",
            "away_score",
            "home_team",
            "home_score"
        ]
    ]
)

Week 11 games used to update Week 12 ratings: 13


,week,gameday,away_team,away_score,home_team,home_score
0,11,2026-11-19,IND,3.0,HOU,26.0
1,11,2026-11-22,ARI,7.0,KC,34.0
2,11,2026-11-22,BAL,33.0,CAR,35.0
3,11,2026-11-22,JAX,18.0,NYG,23.0
4,11,2026-11-22,LV,24.0,DEN,38.0
5,11,2026-11-22,MIA,9.0,BUF,26.0
6,11,2026-11-22,MIN,10.0,SF,35.0
7,11,2026-11-22,NO,13.0,CHI,21.0
8,11,2026-11-22,NYJ,27.0,LAC,24.0
9,11,2026-11-22,PIT,16.0,PHI,19.0


# Convert Prior Week Games to Team Game Rows

Only the most recently completed week is used for the new weekly update.

For Week 3:

- Week 2 team strength is the entering prior
- Week 2 game performance is the new evidence
- Week 1 is not counted again because it is already reflected in the Week 2 ratings

Each game becomes two rows:

- one from the home team's perspective
- one from the away team's perspective

In [5]:
home_rows = pd.DataFrame(
    {
        "game_id": update_games["game_id"],
        "week": update_games["week"],
        "team": update_games["home_team"],
        "opponent": update_games["away_team"],
        "is_home": 1,
        "points_for": update_games["home_score"],
        "points_against": update_games["away_score"]
    }
)

away_rows = pd.DataFrame(
    {
        "game_id": update_games["game_id"],
        "week": update_games["week"],
        "team": update_games["away_team"],
        "opponent": update_games["home_team"],
        "is_home": 0,
        "points_for": update_games["away_score"],
        "points_against": update_games["home_score"]
    }
)

team_games = pd.concat(
    [home_rows, away_rows],
    ignore_index=True
)

team_games["point_margin"] = (
    team_games["points_for"]
    - team_games["points_against"]
)

team_games = team_games.sort_values(
    ["team", "game_id"]
).reset_index(drop=True)

print("Team-game rows:", len(team_games))

display(team_games)

Team-game rows: 26


,game_id,week,team,opponent,is_home,points_for,points_against,point_margin
0,2026_11_ARI_KC,11,ARI,KC,0,7.0,34.0,-27.0
1,2026_11_BAL_CAR,11,BAL,CAR,0,33.0,35.0,-2.0
2,2026_11_MIA_BUF,11,BUF,MIA,1,26.0,9.0,17.0
3,2026_11_BAL_CAR,11,CAR,BAL,1,35.0,33.0,2.0
4,2026_11_NO_CHI,11,CHI,NO,1,21.0,13.0,8.0
5,2026_11_CIN_WAS,11,CIN,WAS,0,20.0,22.0,-2.0
6,2026_11_TEN_DAL,11,DAL,TEN,1,51.0,30.0,21.0
7,2026_11_LV_DEN,11,DEN,LV,1,38.0,24.0,14.0
8,2026_11_TB_DET,11,DET,TB,1,31.0,21.0,10.0
9,2026_11_IND_HOU,11,HOU,IND,1,26.0,3.0,23.0


# Attach Entering Team and Opponent Strength

Each team's performance is evaluated relative to what the model believed before the game was played.

For Week 3 projections, Week 2 ratings represent the information available entering Week 2.

This prevents the opponent adjustment from using future information or the same results that are currently being evaluated.

In [6]:
prior_strength = pd.read_parquet(
    prior_strength_path
).copy()

required_prior_columns = [
    "team",
    "weekly_team_strength",
    "weekly_strength_rank"
]

missing_prior_columns = [
    col
    for col in required_prior_columns
    if col not in prior_strength.columns
]

if missing_prior_columns:
    raise KeyError(
        "Missing prior-week strength columns: "
        + ", ".join(missing_prior_columns)
    )

prior_strength = prior_strength[
    required_prior_columns
].rename(
    columns={
        "weekly_team_strength": "entering_team_strength",
        "weekly_strength_rank": "entering_strength_rank"
    }
)

print(
    f"Week {PRIOR_WEEK} entering ratings loaded:",
    len(prior_strength)
)

display(
    prior_strength.sort_values(
        "entering_strength_rank"
    )
)

Week 11 entering ratings loaded: 32


,team,entering_team_strength,entering_strength_rank
0,BUF,8.151564,1
1,JAX,6.485660,2
2,LA,5.981859,3
3,SF,5.208822,4
4,KC,5.070234,5
5,DET,4.974895,6
6,SEA,4.310075,7
7,CIN,4.292347,8
8,DEN,3.953856,9
9,CHI,3.472897,10


In [7]:
team_games = team_games.merge(
    prior_strength[
        [
            "team",
            "entering_team_strength",
            "entering_strength_rank"
        ]
    ],
    on="team",
    how="left",
    validate="many_to_one"
)

opponent_strength = prior_strength[
    [
        "team",
        "entering_team_strength",
        "entering_strength_rank"
    ]
].rename(
    columns={
        "team": "opponent",
        "entering_team_strength": "opponent_entering_strength",
        "entering_strength_rank": "opponent_entering_rank"
    }
)

team_games = team_games.merge(
    opponent_strength,
    on="opponent",
    how="left",
    validate="many_to_one"
)

display(
    team_games[
        [
            "team",
            "entering_strength_rank",
            "entering_team_strength",
            "opponent",
            "opponent_entering_rank",
            "opponent_entering_strength",
            "points_for",
            "points_against",
            "point_margin"
        ]
    ].sort_values(
        "entering_strength_rank"
    )
)

,team,entering_strength_rank,entering_team_strength,opponent,opponent_entering_rank,opponent_entering_strength,points_for,points_against,point_margin
2,BUF,1,8.151564,MIA,29,-4.840968,26.0,9.0,17.0
11,JAX,2,6.485660,NYG,19,-2.056090,18.0,23.0,-5.0
22,SF,4,5.208822,MIN,11,3.069441,35.0,10.0,25.0
12,KC,5,5.070234,ARI,30,-6.133545,34.0,7.0,27.0
8,DET,6,4.974895,TB,15,1.202707,31.0,21.0,10.0
5,CIN,8,4.292347,WAS,18,-1.373597,20.0,22.0,-2.0
7,DEN,9,3.953856,LV,28,-4.433542,38.0,24.0,14.0
4,CHI,10,3.472897,NO,23,-3.331436,21.0,13.0,8.0
16,MIN,11,3.069441,SF,4,5.208822,10.0,35.0,-25.0
9,HOU,13,1.556349,IND,22,-3.273874,26.0,3.0,23.0


# Expected Margin and Opponent-Adjusted Performance

The weekly update is based on how a team performed relative to expectation.

For each team game:

`performance residual = actual margin - expected margin`

Expected margin uses the difference between the two teams' entering strength ratings plus home field advantage.

This means beating a strong opponent is more informative than producing the same result against a weak opponent.

In [8]:
HOME_FIELD_POINTS = 1.5

team_games["expected_margin"] = (
    team_games["entering_team_strength"]
    - team_games["opponent_entering_strength"]
    + np.where(
        team_games["is_home"] == 1,
        HOME_FIELD_POINTS,
        -HOME_FIELD_POINTS
    )
)

team_games["performance_residual"] = (
    team_games["point_margin"]
    - team_games["expected_margin"]
)

display(
    team_games[
        [
            "team",
            "opponent",
            "entering_team_strength",
            "opponent_entering_strength",
            "point_margin",
            "expected_margin",
            "performance_residual"
        ]
    ].sort_values(
        "performance_residual",
        ascending=False
    )
)

,team,opponent,entering_team_strength,opponent_entering_strength,point_margin,expected_margin,performance_residual
22,SF,MIN,5.208822,3.069441,25.0,3.639381,21.360619
9,HOU,IND,1.556349,-3.273874,23.0,6.330223,16.669777
12,KC,ARI,5.070234,-6.133545,27.0,12.703779,14.296221
6,DAL,TEN,-0.672234,-7.194555,21.0,8.022321,12.977679
18,NYG,JAX,-2.056090,6.485660,5.0,-7.041750,12.041750
19,NYJ,LAC,-6.983012,-4.000654,3.0,-4.482358,7.482358
20,PHI,PIT,-4.148071,1.547719,3.0,-4.195790,7.195790
25,WAS,CIN,-1.373597,4.292347,2.0,-4.165944,6.165944
8,DET,TB,4.974895,1.202707,10.0,5.272188,4.727812
7,DEN,LV,3.953856,-4.433542,14.0,9.887398,4.112602


# Control Extreme Single-Game Results

NFL game margins contain substantial single-game noise.

A blowout should affect team strength, but one extreme result should not completely redefine the team's rating.

Performance residuals are therefore capped before the weekly learning rate is applied.

In [9]:
RESIDUAL_SCALE = 14.0
MAX_WEEKLY_UPDATE = 1.5

team_games["weekly_strength_update"] = (
    MAX_WEEKLY_UPDATE
    * np.tanh(
        team_games["performance_residual"]
        / RESIDUAL_SCALE
    )
)

display(
    team_games[
        [
            "team",
            "opponent",
            "point_margin",
            "expected_margin",
            "performance_residual",
            "weekly_strength_update"
        ]
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

,team,opponent,point_margin,expected_margin,performance_residual,weekly_strength_update
22,SF,MIN,25.0,3.639381,21.360619,1.364544
9,HOU,IND,23.0,6.330223,16.669777,1.246193
12,KC,ARI,27.0,12.703779,14.296221,1.155507
6,DAL,TEN,21.0,8.022321,12.977679,1.093772
18,NYG,JAX,5.0,-7.041750,12.041750,1.044483
19,NYJ,LAC,3.0,-4.482358,7.482358,0.733168
20,PHI,PIT,3.0,-4.195790,7.195790,0.709566
25,WAS,CIN,2.0,-4.165944,6.165944,0.620994
8,DET,TB,10.0,5.272188,4.727812,0.488135
7,DEN,LV,14.0,9.887398,4.112602,0.428384


# Offensive and Defensive Weekly Signals

Overall team strength uses the opponent adjusted margin residual.

We also preserve separate offensive and defensive signals because the weekly game projection model needs scoring information.

These are weekly updates, not cumulative season averages.

Opponent entering strength is used conservatively as context until separate entering offensive and defensive strength ratings are maintained.

In [10]:
league_points_per_team_game = (
    team_games["points_for"].mean()
)

OPPONENT_CONTEXT_RATE = 0.25
OFFENSE_DEFENSE_UPDATE_RATE = 0.15

team_games["raw_offense_performance"] = (
    team_games["points_for"]
    - league_points_per_team_game
)

team_games["raw_defense_performance"] = (
    league_points_per_team_game
    - team_games["points_against"]
)

team_games["adjusted_offense_performance"] = (
    team_games["raw_offense_performance"]
    + (
        OPPONENT_CONTEXT_RATE
        * team_games["opponent_entering_strength"]
    )
)

team_games["adjusted_defense_performance"] = (
    team_games["raw_defense_performance"]
    + (
        OPPONENT_CONTEXT_RATE
        * team_games["opponent_entering_strength"]
    )
)

team_games["weekly_offense_adjustment"] = (
    team_games["adjusted_offense_performance"]
    * OFFENSE_DEFENSE_UPDATE_RATE
)

team_games["weekly_defense_adjustment"] = (
    team_games["adjusted_defense_performance"]
    * OFFENSE_DEFENSE_UPDATE_RATE
)

display(
    team_games[
        [
            "team",
            "opponent",
            "points_for",
            "points_against",
            "opponent_entering_strength",
            "raw_offense_performance",
            "adjusted_offense_performance",
            "weekly_offense_adjustment",
            "raw_defense_performance",
            "adjusted_defense_performance",
            "weekly_defense_adjustment"
        ]
    ].sort_values(
        "team"
    )
)

,team,opponent,points_for,points_against,opponent_entering_strength,raw_offense_performance,adjusted_offense_performance,weekly_offense_adjustment,raw_defense_performance,adjusted_defense_performance,weekly_defense_adjustment
0,ARI,KC,7.0,34.0,5.070234,-16.692308,-15.424749,-2.313712,-10.307692,-9.040134,-1.356020
1,BAL,CAR,33.0,35.0,-3.114481,9.307692,8.529072,1.279361,-11.307692,-12.086313,-1.812947
2,BUF,MIA,26.0,9.0,-4.840968,2.307692,1.097450,0.164618,14.692308,13.482066,2.022310
3,CAR,BAL,35.0,33.0,-2.421892,11.307692,10.702219,1.605333,-9.307692,-9.913165,-1.486975
4,CHI,NO,21.0,13.0,-3.331436,-2.692308,-3.525167,-0.528775,10.692308,9.859449,1.478917
5,CIN,WAS,20.0,22.0,-1.373597,-3.692308,-4.035707,-0.605356,1.692308,1.348908,0.202336
6,DAL,TEN,51.0,30.0,-7.194555,27.307692,25.509054,3.826358,-6.307692,-8.106331,-1.215950
7,DEN,LV,38.0,24.0,-4.433542,14.307692,13.199307,1.979896,-0.307692,-1.416078,-0.212412
8,DET,TB,31.0,21.0,1.202707,7.307692,7.608369,1.141255,2.692308,2.992984,0.448948
9,HOU,IND,26.0,3.0,-3.273874,2.307692,1.489224,0.223384,20.692308,19.873839,2.981076


# Build Week-Level Feature Table

The final feature table contains only the new evidence that should move the prior week's ratings.

For Week 3:

`Week 3 strength = Week 2 strength + Week 2 performance update`

The previous week's rating therefore acts as the prior, and older games are not counted again.

In [11]:
team_features = (
    team_games
    .groupby("team", as_index=False)
    .agg(
        games_in_update=("game_id", "count"),

        entering_team_strength=(
            "entering_team_strength",
            "first"
        ),

        entering_strength_rank=(
            "entering_strength_rank",
            "first"
        ),

        average_opponent_strength=(
            "opponent_entering_strength",
            "mean"
        ),

        average_opponent_rank=(
            "opponent_entering_rank",
            "mean"
        ),

        actual_margin=(
            "point_margin",
            "mean"
        ),

        expected_margin=(
            "expected_margin",
            "mean"
        ),

        performance_residual=(
            "performance_residual",
            "mean"
        ),

        weekly_strength_update=(
            "weekly_strength_update",
            "mean"
        ),

        weekly_offense_adjustment=(
            "weekly_offense_adjustment",
            "mean"
        ),

        weekly_defense_adjustment=(
            "weekly_defense_adjustment",
            "mean"
        )
    )
)

display(
    team_features[
        [
            "team",
            "entering_strength_rank",
            "entering_team_strength",
            "average_opponent_rank",
            "average_opponent_strength",
            "actual_margin",
            "expected_margin",
            "performance_residual",
            "weekly_strength_update",
            "weekly_offense_adjustment",
            "weekly_defense_adjustment"
        ]
    ].sort_values(
        "weekly_strength_update",
        ascending=False
    )
)

,team,entering_strength_rank,entering_team_strength,average_opponent_rank,average_opponent_strength,actual_margin,expected_margin,performance_residual,weekly_strength_update,weekly_offense_adjustment,weekly_defense_adjustment
22,SF,4,5.208822,11.0,3.069441,25.0,3.639381,21.360619,1.364544,1.811258,2.168950
9,HOU,13,1.556349,22.0,-3.273874,23.0,6.330223,16.669777,1.246193,0.223384,2.981076
12,KC,5,5.070234,30.0,-6.133545,27.0,12.703779,14.296221,1.155507,1.316146,2.273838
6,DAL,17,-0.672234,32.0,-7.194555,21.0,8.022321,12.977679,1.093772,3.826358,-1.215950
18,NYG,19,-2.056090,2.0,6.485660,5.0,-7.041750,12.041750,1.044483,0.139366,1.097058
19,NYJ,31,-6.983012,25.0,-4.000654,3.0,-4.482358,7.482358,0.733168,0.346129,-0.196178
20,PHI,27,-4.148071,14.0,1.547719,3.0,-4.195790,7.195790,0.709566,-0.645807,1.211886
25,WAS,18,-1.373597,8.0,4.292347,2.0,-4.165944,6.165944,0.620994,-0.092883,0.714809
8,DET,6,4.974895,15.0,1.202707,10.0,5.272188,4.727812,0.488135,1.141255,0.448948
7,DEN,9,3.953856,28.0,-4.433542,14.0,9.887398,4.112602,0.428384,1.979896,-0.212412


# Sanity Checks

Before saving, verify that:

- all 32 teams are represented
- every team has an entering Week 2 rating
- every opponent has an entering Week 2 rating
- no Week 3 game results were used
- weekly strength updates remain within the intended cap

In [12]:
print("Teams in feature table:", len(team_features))
print(
    "Unique teams:",
    team_features["team"].nunique()
)

print(
    "Missing entering team strengths:",
    team_features[
        "entering_team_strength"
    ].isna().sum()
)

print(
    "Missing opponent strengths:",
    team_features[
        "average_opponent_strength"
    ].isna().sum()
)

print(
    "Largest positive weekly update:",
    round(
        team_features[
            "weekly_strength_update"
        ].max(),
        3
    )
)

print(
    "Largest negative weekly update:",
    round(
        team_features[
            "weekly_strength_update"
        ].min(),
        3
    )
)

if (
    completed_games["week"] >= TARGET_WEEK
).any():
    raise ValueError(
        "DATA LEAKAGE: target-week results "
        "were found in completed games."
    )

if len(team_features) != 32:
    print()
    print(
        "WARNING: Feature table does not "
        "contain all 32 teams."
    )

Teams in feature table: 26
Unique teams: 26
Missing entering team strengths: 0
Missing opponent strengths: 0
Largest positive weekly update: 1.365
Largest negative weekly update: -1.365



# Save In Season Feature Table

This output will be consumed by:

`04_Weekly_Team_Strength.ipynb`

The file contains only the new in season evidence. It does not overwrite or replace the preseason team strength table.


In [13]:
output_path = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

team_features = team_features.sort_values(
    "team"
).reset_index(drop=True)

team_features.to_parquet(
    output_path,
    index=False
)

print("Saved:", output_path)


Saved: ..\..\data\processed\weekly\week_12_inseason_features.parquet


# Interpretation

The weekly model is now sequential.

For Week 3:

1. Week 2 team strength is treated as the entering prior.
2. Only Week 2 game results provide new evidence.
3. Performance is evaluated relative to the opponent's entering strength.
4. Extreme single game residuals are capped.
5. A conservative learning rate determines the weekly rating movement.
6. The resulting update will be added to the Week 2 rating in `04_Weekly_Team_Strength.ipynb`.

This prevents older games from being counted repeatedly and makes strength of opponent part of every weekly update.

The frozen preseason model remains unchanged.